# 👥 Customer Churn Prediction & Retention Analytics
### Purpose: Customer attrition probability, key churn drivers & targeted retention strategies
**Tech Stack:** Scikit-Learn (Logistic Regression, KNN, Random Forest, XGBoost) + Deep Learning (PyTorch / ANN MLP)

Following the project architecture:
1. **Dataset Inspection**: 10,000 retail banking customers across European markets (target `Exited`)
2. **Preventing Target Leakage & ID Removal**: Stripping `RowNumber`, `CustomerId`, `Surname`
3. **Exploratory Data Analysis**: Age distribution, balance thresholds, product counts, and active member status
4. **Preprocessing Pipeline**: Numerical scaling (median imputation) + Categorical one-hot encoding
5. **Model Comparisons**: Logistic Regression, KNN, Random Forest, XGBoost, and Deep Learning ANN
6. **Feature Importance & Insights**: Identifying primary churn drivers (e.g. single-product holding, inactive members)
7. **Model Export**: Save fitted preprocessor and production model to `models/`

In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, f1_score, precision_score, recall_score, roc_curve
)

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded.")

### 1. Data Ingestion & Identifiers Removal (Preventing Leakage)

In [ ]:
DATA_PATH = Path("../data/raw/churn.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/churn.csv")

df = pd.read_csv(DATA_PATH)
print("Churn Dataset Shape:", df.shape)

# Drop non-predictive identifiers to avoid false correlation
leakage_ids = ["RowNumber", "CustomerId", "Surname"]
X = df.drop(columns=[c for c in leakage_ids if c in df.columns] + ["Exited"])
y = df["Exited"]

print(f"Features: {X.columns.tolist()}")
print(f"Churn Prevalence: {y.mean():.2%}")

### 2. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="Age", hue="Exited", multiple="stack", ax=axes[0], palette=["#00C076", "#F59E0B"])
axes[0].set_title("Customer Age vs Churn Distribution")

sns.countplot(data=df, x="NumOfProducts", hue="Exited", ax=axes[1], palette=["#00C076", "#F59E0B"])
axes[1].set_title("Number of Products vs Churn")
plt.tight_layout()
plt.show()

### 3. Preprocessing Pipeline & Stratified Split

In [ ]:
num_cols = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
cat_cols = ["Geography", "Gender", "HasCrCard", "IsActiveMember"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)
print(f"X_train_t: {X_train_t.shape}, X_test_t: {X_test_t.shape}")

### 4. Training Churn ML & Deep Learning ANN Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=42),
    "KNN (k=7)": KNeighborsClassifier(n_neighbors=7),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight="balanced", n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=120, max_depth=5, learning_rate=0.08, eval_metric="logloss", random_state=42),
    "Deep Learning ANN (MLP)": MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", max_iter=40, random_state=42, early_stopping=True)
}

leaderboard = []

for name, model in models.items():
    print(f"Training Churn Model: {name}...")
    t0 = time.time()
    model.fit(X_train_t, y_train)
    train_time = time.time() - t0
    
    t1 = time.time()
    preds = model.predict(X_test_t)
    latency = ((time.time() - t1) / len(X_test_t)) * 1000
    
    probs = model.predict_proba(X_test_t)[:, 1] if hasattr(model, "predict_proba") else None
    
    leaderboard.append({
        "Module": "Customer Churn",
        "Model": name,
        "F1-Score": round(f1_score(y_test, preds), 4),
        "ROC-AUC": round(roc_auc_score(y_test, probs) if probs is not None else 0.0, 4),
        "PR-AUC": round(average_precision_score(y_test, probs) if probs is not None else 0.0, 4),
        "Precision": round(precision_score(y_test, preds), 4),
        "Recall": round(recall_score(y_test, preds), 4),
        "Latency (ms)": round(latency, 3),
        "Train Time (s)": round(train_time, 2)
    })

df_leaderboard = pd.DataFrame(leaderboard).sort_values("F1-Score", ascending=False)
display(df_leaderboard)

### 5. Churn Drivers & Retention Strategy Insights

In [ ]:
best_model = models["Random Forest"]
cat_names = preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(cat_cols)
feature_names = num_cols + list(cat_names)

importances = pd.Series(best_model.feature_importances_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(9, 4.5))
importances.head(8).plot(kind="barh", color="#F59E0B")
plt.title("Primary Drivers of Customer Attrition (Random Forest Gini Importance)")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 6. Export Production Churn Model Artifacts

In [ ]:
out_dir = Path("../models") if Path("../models").exists() else Path("models")
joblib.dump(preprocessor, out_dir / "churn_preprocessor.joblib")
joblib.dump(best_model, out_dir / "churn_model.joblib")
print(f"Exported churn artifacts to {out_dir}")